# Lesson 3: Classification Pipeline

**Duration:** 90 minutes

## Learning Objectives
- Create and manage data loaders for efficient batch processing
- Implement a complete training loop with loss computation
- Validate models on held-out data
- Test and perform inference on new data
- Track training progress and metrics

## 3.1 Data Loading and Batching

DataLoaders efficiently load data in batches during training:

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision import datasets
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# Define transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

# Load CIFAR-10 dataset
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# Create data loaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f'Total training samples: {len(train_dataset)}')
print(f'Total test samples: {len(test_dataset)}')
print(f'Number of training batches: {len(train_loader)}')
print(f'Number of test batches: {len(test_loader)}')

# Verify data
images, labels = next(iter(train_loader))
print(f'\nBatch shape: {images.shape}')
print(f'Labels shape: {labels.shape}')
print(f'Class distribution in batch: {torch.bincount(labels)}')
print(f'Label mapping: 0=Airplane, 1=Automobile, 2=Bird, etc.')

## 3.2 Define Model, Loss, and Optimizer

Set up the network architecture, loss function, and optimization algorithm:

In [ ]:
# Define model
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 10)
        
    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(-1, 64 * 8 * 8)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleCNN().to(device)
print(f'Model sent to device: {device}')
print(f'Total parameters: {sum(p.numel() for p in model.parameters())}')

# Define loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f'Loss function: CrossEntropyLoss')
print(f'Optimizer: Adam with lr=0.001')

## 3.3 Training Loop

Train the model on the training data:

In [ ]:
# Training parameters
num_epochs = 3
train_losses = []
train_accuracies = []

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}'):
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Track metrics
        epoch_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    avg_loss = epoch_loss / len(train_loader)
    accuracy = correct / total
    train_losses.append(avg_loss)
    train_accuracies.append(accuracy)
    
    print(f'Epoch [{epoch+1}/{num_epochs}] Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}')

print('\nTraining completed!')
print(f'Final training accuracy: {train_accuracies[-1]:.4f}')

## 3.4 Validation and Testing

Evaluate the model on test data:

In [ ]:
# Test the model
model.eval()
test_correct = 0
test_total = 0
all_predictions = []
all_labels = []

with torch.no_grad():  # Disable gradient computation for inference
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()
        
        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_accuracy = test_correct / test_total
print(f'Test Accuracy: {test_accuracy:.4f}')
print(f'Test Correct: {test_correct}/{test_total}')

# Per-class accuracy
from sklearn.metrics import classification_report
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
print('\nPer-class Performance:')
print(classification_report(all_labels, all_predictions, target_names=class_names))

# Plot training curves
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(train_losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(train_accuracies)
plt.axhline(y=test_accuracy, color='r', linestyle='--', label=f'Test Acc: {test_accuracy:.4f}')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training Accuracy')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 3.5 Inference on New Data

Make predictions on individual samples:

In [ ]:
# Get some test images
images, labels = next(iter(test_loader))
images = images.to(device)

# Make predictions
with torch.no_grad():
    outputs = model(images[:8])
    probabilities = torch.softmax(outputs, dim=1)
    predictions = torch.argmax(probabilities, dim=1)

# Visualize
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    img = images[i].cpu().transpose(0, 2).transpose(0, 1)
    ax.imshow((img * 0.5 + 0.5).numpy())  # Denormalize for visualization
    pred_class = predictions[i].item()
    true_class = labels[i].item()
    confidence = probabilities[i, pred_class].item()
    
    color = 'green' if pred_class == true_class else 'red'
    ax.set_title(f'Pred: {class_names[pred_class]}\nTrue: {class_names[true_class]}\nConf: {confidence:.2f}', color=color)
    ax.axis('off')

plt.tight_layout()
plt.show()